In [1]:
# DS 340H: Assignment Week 7
# Date: 3/2/2026
# Author: Celeste Deudon

import pandas as pd

In [5]:
# Load from CSV file
inspectionsData = pd.read_csv('inspectionsData.csv')

C:\Users\celes\AppData\Local\Temp\ipykernel_13308\1165627864.py:2: DtypeWarning: Columns (0: zip) have mixed types. Specify dtype option on import or set low_memory=False.
  inspectionsData = pd.read_csv('inspectionsData.csv')


In [6]:
# ==========================================
# STEP 1: DATA EXPLORATION
# ==========================================

#print(inspectionsData.columns)
# ['businessname', 'dbaname', 'legalowner', 'namelast', 'namefirst',
#       'licenseno', 'issdttm', 'expdttm', 'licstatus', 'licensecat',      
#       'descript', 'result', 'resultdttm', 'violation', 'viol_level',     
#       'violdesc', 'violdttm', 'viol_status', 'status_date', 'comments',  
#       'address', 'city', 'state', 'zip', 'property_id', 'location']

#print(inspectionsData.head())

# Look for inconsistent/disparities in variables
print(inspectionsData["city"].value_counts()) 
    # Clean the titles if they contain "/"
        # Ex: "FINANCIAL DISTRICT/" & "FINANCIAL DISTRICT"
    # Combine some of the cities where titles are off
        # Ex: "DOWNTOWN/FINANCIAL DISTRICT" & Above
print(inspectionsData["result"].value_counts()) 
    # Add "Fail" as a "HE_Fail" and "Pass" as "HE_Pass"

city
BOSTON                         388625
DORCHESTER                     116963
ROXBURY                         67530
EAST BOSTON                     52597
JAMAICA PLAIN                   41246
ALLSTON                         36877
SOUTH BOSTON                    29827
BRIGHTON                        29090
ROSLINDALE                      24505
WEST ROXBURY                    23124
HYDE PARK                       16235
MATTAPAN                        16022
MISSION HILL                    15316
CHARLESTOWN                     10991
CHESTNUT HILL                     749
MISSION HILL/                     597
SOUTH BOSTON/                     394
                                  223
ROSLINDALE/                       212
FENWAY                            142
FINANCIAL DISTRICT                130
FINANCIAL DISTRICT/               126
CHARLESTOWN/                      121
WEST ROXBURY//                     82
ROXBURY/BOSTON                     62
DOWNTOWN/FINANCIAL DISTRICT        51
BRIGHTO

In [11]:
# ==========================================
# STEP 2: DATA CLEANING
# ==========================================

# All of the steps below are simply a reflection/mirror of what is 
    # explained in the analysis plan submitted in the write up.

# ── 1. Parse dates & extract year ─────────────────────────────────────────────
# Decision: If values un-parsable, make them NaT
inspectionsData['resultdttm'] = pd.to_datetime(inspectionsData['resultdttm'], errors='coerce')

# Extract year (switch to .dt.to_period('Q') later if we want quarter-year)
inspectionsData['year'] = inspectionsData['resultdttm'].dt.year
inspectionsData['year'] = inspectionsData['year'].astype(int) # Cause it was in 2025.0 format

# print("Rows with missing resultdttm after parsing:", inspectionsData['resultdttm'].isna().sum()) 
# No missing rows, no 0s


Rows with missing resultdttm after parsing: 0


In [12]:
# ── 2. Explore missing values ──────────────────────────────────────────────────

# Show count and % of missing values for every column = Thank you Claude for this code
missing = pd.DataFrame({
    'missing_count': inspectionsData.isna().sum(),
    'missing_pct':   inspectionsData.isna().sum() / len(inspectionsData) * 100
}).sort_values('missing_pct', ascending=False)

print(missing)

# Decisions per column:
# - viol_level, violdesc, viol_status: missing = no violation on that row (Pass rows) 
# XXXX^
# - zip: small number missing for zip, but none for city -> so no major problem

              missing_count  missing_pct
dbaname              845983    99.061705
status_date          494707    57.928491
namefirst            364145    42.640129
legalowner           296638    34.735292
property_id          152588    17.867531
location              76148     8.916669
comments              69299     8.114675
violdesc              52171     6.109045
violdttm              45847     5.368526
viol_level            45845     5.368292
violation             45845     5.368292
viol_status           45845     5.368292
state                  1122     0.131382
issdttm                 759     0.088876
expdttm                 630     0.073771
zip                     603     0.070609
address                 180     0.021077
namelast                  0     0.000000
businessname              0     0.000000
descript                  0     0.000000
licensecat                0     0.000000
licstatus                 0     0.000000
licenseno                 0     0.000000
resultdttm      

In [14]:
# ── 3. Reconstruct to a inspection-level data ──────────────────────────────────────

# Each row is one violation aka group by business + inspection date to get one row per inspection
inspection_df = (
    inspectionsData.groupby(['businessname', 'resultdttm', 'city', 'zip', 'year', 'result', 'descript'])
    .agg(
        num_violations = ('violation', 'count')   # count violations per inspection, add in new col
    )
    .reset_index()
)

print(inspection_df.shape)
print(inspection_df.head()) # Looks goooood! 

(198820, 8)
                               businessname                resultdttm  \
0                       10-Eleven Food Shop 2025-10-17 14:39:37+00:00   
1                       10-Eleven Food Shop 2025-10-21 19:14:28+00:00   
2  100 Federal Market & Barista- 10th Floor 2023-08-23 12:28:50+00:00   
3  100 Federal Market & Barista- 10th Floor 2024-03-11 16:18:18+00:00   
4  100 Federal Market & Barista- 10th Floor 2025-09-30 14:45:23+00:00   

         city    zip  year   result           descript  num_violations  
0  ROSLINDALE  02131  2025  HE_Fail        Retail Food               5  
1  ROSLINDALE  02131  2025  HE_Pass        Retail Food               5  
2      BOSTON  02110  2023  HE_Pass  Eating & Drinking               0  
3      BOSTON  02110  2024  HE_Pass  Eating & Drinking               0  
4      BOSTON  02110  2025  HE_Pass  Eating & Drinking               0  


In [17]:
# ── Added City Cleaning as seen problematic from Exploration ──────────────────────────────────

print("\nCity levels BEFORE cleaning:\n", inspection_df['city'].value_counts())

# Strip whitespace and uppercase everything for consistency since we cannot see those always (just in case)
inspection_df['city'] = inspection_df['city'].str.strip().str.upper()

# Remove trailing slashes (like the example we found above "FINANCIAL DISTRICT/" → "FINANCIAL DISTRICT")
inspection_df['city'] = inspection_df['city'].str.rstrip('/')

# Manually combine neighborhood names that refer to the same place
city_map = {
    'DOWNTOWN/FINANCIAL DISTRICT': 'FINANCIAL DISTRICT',
    'DOWNTOWN': 'FINANCIAL DISTRICT',
    'ROXBURY/BOSTON': 'ROXBURY',
    'BOSTON/WEST END': 'WEST END',
    'BOSTON/CHINATOWN': 'CHINATOWN',
}
inspection_df['city'] = inspection_df['city'].replace(city_map)

print("\nCity levels AFTER cleaning:\n", inspection_df['city'].value_counts())


City levels BEFORE cleaning:
 city
BOSTON                96384
DORCHESTER            24010
ROXBURY               13728
EAST BOSTON           12988
JAMAICA PLAIN          7522
ALLSTON                7497
BRIGHTON               7253
SOUTH BOSTON           6315
ROSLINDALE             5210
WEST ROXBURY           4519
HYDE PARK              3587
MATTAPAN               3293
CHARLESTOWN            3266
MISSION HILL           2800
CHESTNUT HILL           165
FINANCIAL DISTRICT      144
FENWAY                   61
ROXBURY/BOSTON           20
BACK BAY                 14
DORCHESTER CENTER        11
BOSTON/WEST END          10
ROXBURY CROSSIN          10
SOUTH END                 8
BOSTON/CHINATOWN          5
Name: count, dtype: int64

City levels AFTER cleaning:
 city
BOSTON                96384
DORCHESTER            24010
ROXBURY               13748
EAST BOSTON           12988
JAMAICA PLAIN          7522
ALLSTON                7497
BRIGHTON               7253
SOUTH BOSTON           6315
ROSLIND

In [19]:
# ── 4. Clean and standardize the result column ────────────────────────────────

print("\nResult levels BEFORE cleaning:\n", inspection_df['result'].value_counts(dropna=False))

# Combine "Pass"/"Fail" with the "HE_Pass"/"HE_Fail" entries — same outcome, different recording
inspection_df['result'] = inspection_df['result'].replace({
    'Pass': 'HE_Pass',
    'Fail': 'HE_Fail',
})

print("\nResult levels AFTER cleaning:\n", inspection_df['result'].value_counts(dropna=False))

# Turns out with the cleaning we have already done it is not needed! 


Result levels BEFORE cleaning:
 result
HE_Pass       86710
HE_Fail       61965
HE_Filed      24926
HE_NotReq      9927
HE_FailExt     9496
HE_Hearing     3223
HE_OutBus      1051
HE_TSOP         817
HE_VolClos      476
HE_Closure      115
HE_Misc          76
HE_FAILNOR       21
DATAERR           8
HE_Hold           5
Closed            2
NoViol            1
Failed            1
Name: count, dtype: int64

Result levels AFTER cleaning:
 result
HE_Pass       86710
HE_Fail       61965
HE_Filed      24926
HE_NotReq      9927
HE_FailExt     9496
HE_Hearing     3223
HE_OutBus      1051
HE_TSOP         817
HE_VolClos      476
HE_Closure      115
HE_Misc          76
HE_FAILNOR       21
DATAERR           8
HE_Hold           5
Closed            2
NoViol            1
Failed            1
Name: count, dtype: int64


In [20]:
# ── 5. Add numeric severity column ────────────────────────────────────────────

print("\nviol_level levels:\n", inspection_df['viol_level'].value_counts(dropna=False))

# Create a new numeric column — keeping the original viol_level column intact
# * = 1 (low severity), ** = 2 (medium), *** = 3 (high)
inspection_df['viol_level_num'] = inspection_df['viol_level'].replace({'*': 1, '**': 2, '***': 3})

# Any value that wasn't *, **, or *** (including NaN) stays as NaN
inspection_df['viol_level_num'] = pd.to_numeric(inspection_df['viol_level_num'], errors='coerce')


KeyError: 'viol_level'